In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

In [ ]:
ROWS = 5
COLS = 5

START = (0, 1)
GOAL = (4, 3)

OBSTACLES = [
    (1, 1),
    (1, 3),
    (3, 2)
]

In [ ]:
actions = {
    0: "UP",
    1: "DOWN",
    2: "LEFT",
    3: "RIGHT"
}

In [ ]:
def state_number(state):
    row, col = state
    return row * COLS + col

In [ ]:
def move(state, action):

    row, col = state

    new_row = row
    new_col = col

    if action == 0:
        new_row -= 1

    elif action == 1:
        new_row += 1

    elif action == 2:
        new_col -= 1

    elif action == 3:
        new_col += 1


    # Boundary check
    if new_row < 0 or new_row >= ROWS or \
       new_col < 0 or new_col >= COLS:

        return state, -10, False


    new_state = (new_row, new_col)


    # Obstacle check
    if new_state in OBSTACLES:

        return state, -10, False


    # Goal check
    if new_state == GOAL:

        return new_state, 100, True


    # Normal movement
    return new_state, -1, False

In [ ]:
Q = np.zeros((25, 4))

print(Q)

In [ ]:
alpha = 0.1
gamma = 0.9
epsilon = 0.2

episodes = 300

In [ ]:
total_rewards = []

for episode in range(episodes):

    state = START

    episode_reward = 0

    for step in range(100):

        state_idx = state_number(state)


        # Choose action
        if random.random() < epsilon:

            # Explore
            action = random.randint(0, 3)

        else:

            # Exploit
            action = np.argmax(Q[state_idx])


        # Perform action
        next_state, reward, is_terminal = move(
            state,
            action
        )


        next_state_idx = state_number(next_state)


        # Q-learning formula
        Q[state_idx, action] = Q[state_idx, action] + alpha * (
            reward
            + gamma * np.max(Q[next_state_idx])
            - Q[state_idx, action]
        )


        state = next_state

        episode_reward += reward


        if is_terminal:
            break


    total_rewards.append(episode_reward)

In [ ]:
state = START

optimal_path = [state]

for step in range(30):

    if state == GOAL:
        break

    state_idx = state_number(state)

    action = np.argmax(Q[state_idx])

    next_state, reward, is_terminal = move(
        state,
        action
    )

    optimal_path.append(next_state)

    state = next_state

    if is_terminal:
        break


print("Learned Path:")
print(optimal_path)

In [ ]:
from matplotlib.colors import ListedColormap

def show_path(path, start, goal, obstacles, title):

    # 0 = free
    # 1 = obstacle
    # 2 = start
    # 3 = goal

    maze = np.zeros((ROWS, COLS))

    for r, c in obstacles:
        maze[r, c] = 1

    maze[start] = 2
    maze[goal] = 3

    cmap = ListedColormap([
        "white",       # free cell
        "gray",        # obstacle
        "lightblue",   # start
        "lightgreen"   # goal
    ])

    plt.figure(figsize=(6, 6))

    plt.imshow(
        maze,
        cmap=cmap,
        vmin=0,
        vmax=3
    )

    # Grid
    plt.xticks(np.arange(-0.5, COLS, 1), [])
    plt.yticks(np.arange(-0.5, ROWS, 1), [])

    plt.grid(
        color="black",
        linewidth=1
    )

    # Obstacles
    for r, c in obstacles:
        plt.text(
            c, r, "X",
            ha="center",
            va="center",
            fontsize=18,
            fontweight="bold"
        )

    # Start
    plt.text(
        start[1],
        start[0],
        "S",
        ha="center",
        va="center",
        fontsize=15,
        fontweight="bold"
    )

    # Goal
    plt.text(
        goal[1],
        goal[0],
        "G",
        ha="center",
        va="center",
        fontsize=15,
        fontweight="bold"
    )

    # Path
    rows = [p[0] for p in path]
    cols = [p[1] for p in path]

    plt.plot(
        cols,
        rows,
        marker="o",
        linewidth=2
    )

    # Number the path
    for i, (r, c) in enumerate(path):

        if (r, c) != start and (r, c) != goal:

            plt.text(
                c,
                r,
                str(i),
                ha="center",
                va="center",
                fontsize=9
            )

    plt.title(title)
    plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(total_rewards)

plt.xlabel("Episode")
plt.ylabel("Total Reward")

plt.title("Learning Progress")

plt.grid()

plt.show()

In [ ]:
arrow = {
    0: "\u2191",
    1: "\u2193",
    2: "\u2190",
    3: "\u2192"
}

print("Learned Policy\n")

for r in range(ROWS):

    for c in range(COLS):

        state = (r, c)

        if state == START:
            print(" S ", end=" ")

        elif state == GOAL:
            print(" G ", end=" ")

        elif state in OBSTACLES:
            print(" X ", end=" ")

        else:
            state_idx = state_number(state)
            best_action = np.argmax(Q[state_idx])

            print(
                f" {arrow[best_action]} ",
                end=" "
            )

    print()

In [ ]:
NEW_START = (2, 0)

In [ ]:
def test_policy(start, goal, obstacles, max_steps=30):

    state = start

    path = [state]

    for step in range(max_steps):

        if state == goal:
            break

        state_idx = state_number(state)

        action = np.argmax(Q[state_idx])

        row, col = state

        new_row = row
        new_col = col

        if action == 0:
            new_row -= 1

        elif action == 1:
            new_row += 1

        elif action == 2:
            new_col -= 1

        elif action == 3:
            new_col += 1


        # Boundary check
        if new_row < 0 or new_row >= ROWS or \
           new_col < 0 or new_col >= COLS:

            next_state = state

        else:

            next_state = (
                new_row,
                new_col
            )

            if next_state in obstacles:
                next_state = state


        state = next_state

        path.append(state)


    return path

In [ ]:
path_new_start = test_policy(
    NEW_START,
    GOAL,
    OBSTACLES
)

print("Path from new start:")
print(path_new_start)

In [ ]:
show_path(
    path_new_start,
    NEW_START,
    GOAL,
    OBSTACLES,
    "Same Maze - Different Start State"
)

In [ ]:
NEW_GOAL = (0, 4)

path_new_goal = test_policy(
    START,
    NEW_GOAL,
    OBSTACLES
)

print("Path with new goal:")
print(path_new_goal)

In [ ]:
show_path(
    path_new_goal,
    START,
    NEW_GOAL,
    OBSTACLES,
    "Same Maze - Different Goal"
)

In [ ]:
if path_new_goal[-1] == NEW_GOAL:
    print("Agent reached the NEW goal.")
else:
    print("Agent FAILED to reach the NEW goal.")